# Assignment A2 — Random Hill-Climbing

## Algorithm: Random Hill-Climbing (RHC)

**Steps:**
1. Choose a string at random. Call this string `best_evaluated`.
2. Choose a locus at random to flip. If the flip leads to an equal or higher fitness, set `best_evaluated` to the resulting string.
3. Go to step 2 until a maximum number of evaluations have been performed.
4. Return the current value of `best_evaluated`.

**Parameter:** `k` = number of fitness evaluations (budget)

In [1]:
import random
import time

def load_data(file_name):
    items, max_W = [], 0
    with open(file_name, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
        n = int(lines[0])
        for i in range(1, n + 1):
            parts = lines[i].split()
            items.append((int(parts[1]), int(parts[2])))
        max_W = int(lines[n + 1])
    return items, max_W

def verify_quality(solution, items, W):
    total_weight, total_value = 0, 0
    for i, bit in enumerate(solution):
        if bit == 1:
            total_weight += items[i][0]
            total_value  += items[i][1]
    return 0 if total_weight > W else total_value

def generate_random_solution(n):
    return [random.randint(0, 1) for _ in range(n)]

In [2]:
def random_hill_climbing(k, items, W):
    """Random Hill-Climbing: k = max number of fitness evaluations."""
    n = len(items)
    best = generate_random_solution(n)
    best_val = verify_quality(best, items, W)
    evaluations = 1

    while evaluations < k:
        locus = random.randint(0, n - 1)
        neighbour = best[:]
        neighbour[locus] ^= 1                   # flip one bit
        neighbour_val = verify_quality(neighbour, items, W)
        evaluations += 1

        if neighbour_val >= best_val:          
            best, best_val = neighbour, neighbour_val

    return best, best_val

In [3]:
def run_experiments(instance_file, k_values, num_runs=5):
    items, W = load_data(instance_file)
    n = len(items)
    output_filename = f"results_rhc_n{n}.txt"

    with open(output_filename, 'w') as f:
        f.write(f"--- RHC Results for {instance_file} (n={n}, W={W}) ---\n")

        for k in k_values:
            f.write(f"\nTesting k = {k} ({num_runs} runs)\n")
            f.write("-" * 30 + "\n")
            total_quality = 0

            for run in range(num_runs):
                start_time = time.time()
                _, best_val = random_hill_climbing(k, items, W)
                elapsed = time.time() - start_time
                total_quality += best_val
                f.write(f"Run {run + 1}: Quality = {best_val}, Time = {elapsed:.4f}s\n")

            avg_quality = total_quality / num_runs
            f.write(f">> Average Quality for k={k}: {avg_quality:.2f}\n")

    print(f"Done! Results saved to {output_filename}")

In [4]:
k_values = [100, 1000, 10000, 100000]

run_experiments('../data/knapsack-20.txt',  k_values, num_runs=5)
run_experiments('../data/knapsack-200.txt', k_values, num_runs=5)

Done! Results saved to results_rhc_n20.txt
Done! Results saved to results_rhc_n200.txt


## Assignment A2 — Steepest Ascent Hill-Climbing

### Algorithm: Steepest Ascent Hill-Climbing (SAHC)

**Steps:**
1. Choose a string at random. Call this string `current_hilltop`.
2. Going from left to right, systematically flip each bit in the string, one at a time, recording the fitness of each one-bit mutant.
3. If any one-bit mutant gives a fitness increase, set `current_hilltop` to the mutant with the highest fitness increase. If there are ties, choose one at random.
4. If there is no fitness increase, save `current_hilltop` and restart from a new random string.
5. When a set number of function evaluations has been performed, return the highest hilltop found.

**Parameter:** `k` = number of fitness evaluations (budget)

In [5]:
def steepest_ascent_hill_climbing(k, items, W):
    """Steepest Ascent Hill-Climbing with random restarts."""
    n = len(items)
    evaluations = 0

    current_hilltop = generate_random_solution(n)
    current_value = verify_quality(current_hilltop, items, W)
    best_solution = current_hilltop[:]
    best_value = current_value

    while evaluations < k:
        best_neighbours = []
        best_neighbour_value = current_value

        for i in range(n):
            if evaluations >= k:
                break

            neighbour = current_hilltop[:]
            neighbour[i] ^= 1
            neighbour_value = verify_quality(neighbour, items, W)
            evaluations += 1

            if neighbour_value > best_neighbour_value:
                best_neighbours = [neighbour]
                best_neighbour_value = neighbour_value
            elif neighbour_value == best_neighbour_value and neighbour_value > current_value:
                best_neighbours.append(neighbour)

        if best_neighbour_value > current_value:
            current_hilltop = random.choice(best_neighbours)
            current_value = best_neighbour_value
            if current_value > best_value:
                best_solution = current_hilltop[:]
                best_value = current_value
        else:
            if current_value > best_value:
                best_solution = current_hilltop[:]
                best_value = current_value
            current_hilltop = generate_random_solution(n)
            current_value = verify_quality(current_hilltop, items, W)

    return best_solution, best_value

In [ ]:
def run_sahc_experiments(instance_file, k_values, num_runs=5):
    items, W = load_data(instance_file)
    n = len(items)
    output_filename = f"results_sahc_n{n}.txt"

    with open(output_filename, 'w') as f:
        f.write(f"--- SAHC Results for {instance_file} (n={n}, W={W}) ---\n")

        for k in k_values:
            f.write(f"\nTesting k = {k} ({num_runs} runs)\n")
            f.write("-" * 30 + "\n")
            total_quality = 0

            for run in range(num_runs):
                start_time = time.time()
                _, best_val = steepest_ascent_hill_climbing(k, items, W)
                elapsed = time.time() - start_time
                total_quality += best_val
                f.write(f"Run {run + 1}: Quality = {best_val}, Time = {elapsed:.4f}s\n")

            avg_quality = total_quality / num_runs
            f.write(f">> Average Quality for k={k}: {avg_quality:.2f}\n")

    print(f"Done! Results saved to {output_filename}")

run_sahc_experiments('../data/knapsack-20.txt', [100, 1000, 10000, 100000], num_runs=5)
run_sahc_experiments('../data/knapsack-200.txt', [100, 1000, 10000, 100000], num_runs=5)

Done! Results saved to results_sahc_n20.txt
Done! Results saved to results_sahc_n200.txt


## SAHC Report

### 1. Algorithm Used
We used the **Steepest Ascent Hill-Climbing (SAHC)** algorithm with random restarts. For each current solution, all one-bit neighbours are evaluated, and the best improving neighbour is selected. If no improving neighbour exists, the algorithm restarts from a new random solution.

### 2. Parameter Settings
- Problem instances: `n = 20` and `n = 200`
- Parameter tested: `k = 100, 1000, 10000, 100000`
- Number of runs: `5`

### 3. Comparative Results

#### Instance size: n = 20

| Parameter (k) | Run 1 | Run 2 | Run 3 | Run 4 | Run 5 | Average Quality |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| k = 100 | 609 | 692 | 571 | 770 | 599 | 648.20 |
| k = 1000 | 723 | 722 | 709 | 709 | 714 | 715.40 |
| k = 10000 | 785 | 776 | 787 | 785 | 776 | 781.80 |
| k = 100000 | 787 | 787 | 787 | 787 | 787 | 787.00 |

#### Instance size: n = 200

| Parameter (k) | Run 1 | Run 2 | Run 3 | Run 4 | Run 5 | Average Quality |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| k = 100 | 0 | 0 | 0 | 96238 | 0 | 19247.60 |
| k = 1000 | 0 | 0 | 96678 | 0 | 0 | 19335.60 |
| k = 10000 | 96218 | 97130 | 96728 | 96378 | 96804 | 96651.60 |
| k = 100000 | 97346 | 97248 | 97934 | 97140 | 97147 | 97363.00 |

### 4. Discussion of Results
For both instances, increasing `k` improved the average quality of the final solution. For `n = 20`, the improvement is steady, from `648.20` at `k = 100` to `787.00` at `k = 100000`, and the results become very stable for large `k`. In fact, at `k = 100000`, all 5 runs reached the same quality `787`, which suggests that SAHC is consistently finding the same strong hilltop on the small instance.

For `n = 200`, the effect of `k` is much stronger. At `k = 100` and `k = 1000`, most runs returned `0`, meaning the algorithm often stayed in infeasible regions or did not have enough evaluations to climb to a good feasible solution. Starting from `k = 10000`, all runs became good and stable, with average quality `96651.60`. At `k = 100000`, the results improved further to `97363.00`, showing that a larger evaluation budget helps SAHC exploit the search space much better on the larger instance.

Overall, SAHC performs much better when the evaluation budget is high enough. On the small instance it converges quickly, while on the large instance it needs many more evaluations before the hill-climbing with restarts becomes reliable.